# Asocijativna pravila

## 1. Jednostavni primjer - različite metrike

In [1]:
transactions = [
    {"kruh", "mlijeko", "maslac"},
    {"kruh", "kava"},
    {"kruh", "krafna"},
    {"kruh", "čokolada"},
    {"kruh", "mlijeko"},
    {"kruh", "kava", "krafna"},
    {"kruh", "jogurt"},
    {"kruh", "sir"},
    {"kruh", "čaj"},
    {"kruh", "kava"},
]

N = len(transactions)

def support(itemset):
    count = sum(itemset.issubset(t) for t in transactions)
    return count / N

def confidence(A, B):
    return support(A | B) / support(A)

def lift(A, B):
    return support(A | B) / (support(A) * support(B))

def leverage(A, B):
    return support(A | B) - support(A) * support(B)

def conviction(A, B):
    conf = confidence(A, B)

    if conf == 1:
        return float("inf")

    return (1 - support(B)) / (1 - conf)


A = {"kava"}
B = {"kruh"}

print(f"SUP(kava)           = {support(A):.2f}")
print(f"SUP(kruh)           = {support(B):.2f}")
print(f"SUP(kava ∪ kruh)    = {support(A | B):.2f}")

print()

print(f"CONF(kava -> kruh)  = {confidence(A, B):.2f}")
print(f"CONF(kruh -> kava)  = {confidence(B, A):.2f}")
print(f"LIFT(kava -> kruh)  = {lift(A, B):.2f}")
print(f"LEVERAGE            = {leverage(A, B):.2f}")
print(f"CONVICTION A->B         = {conviction(A, B):.2f}")
print(f"CONVICTION B->A         = {conviction(B, A):.2f}")


SUP(kava)           = 0.30
SUP(kruh)           = 1.00
SUP(kava ∪ kruh)    = 0.30

CONF(kava -> kruh)  = 1.00
CONF(kruh -> kava)  = 0.30
LIFT(kava -> kruh)  = 1.00
LEVERAGE            = 0.00
CONVICTION A->B         = inf
CONVICTION B->A         = 1.00


*Za kava->kruh uočiti: CONF = 1, ALI LIFT = 1 pa je očito da su to nezavisni itemi, a slučaj je da je kruh inače svugdje prisutan pa zapravo ne daje nikakvu korisnu info. Conviction=inf je ovdje malo "čudan", no conviction zna biti numerički nestabilan kad pravilo nikad ne griješi na jako malom datasetu.*

*CONF i CONVICTION nisu simetrični*

In [2]:
transactions.append({"kavijar", "šampanjac"})
A = {"kavijar"}
B = {"šampanjac"}

print(f"SUP(kavijar)           = {support(A):.4f}")
print(f"SUP(šampanjac)           = {support(B):.4f}")
print(f"SUP(kavijar ∪ šampanjac)    = {support(A | B):.4f}")

print()

print(f"CONF(kavijar -> šampanjac)  = {confidence(A, B):.2f}")
print(f"LIFT(kavijar -> šampanjac)  = {lift(A, B):.2f}")
print(f"LEVERAGE            = {leverage(A, B):.2f}")
print(f"CONVICTION  A->B        = {conviction(A, B):.2f}")


SUP(kavijar)           = 0.1000
SUP(šampanjac)           = 0.1000
SUP(kavijar ∪ šampanjac)    = 0.1000

CONF(kavijar -> šampanjac)  = 1.00
LIFT(kavijar -> šampanjac)  = 10.00
LEVERAGE            = 0.09
CONVICTION  A->B        = inf


*"Savršeno" pravilo (CONF, LIFT, LEV, CONV), ALI statistički nebitno (SUP)*

In [3]:
transactions.extend([
    {"veganski_proizvod"},
    {"veganski_proizvod"},
    {"meso"},
    {"meso"},
])

A = {"veganski_proizvod"}
B = {"meso"}

print(f"SUP(veg)           = {support(A):.4f}")
print(f"SUP(meso)           = {support(B):.4f}")
print(f"SUP(veg ∪ meso)    = {support(A | B):.4f}")

print()

print(f"CONF(veg -> meso)  = {confidence(A, B):.4f}")
print(f"LIFT(veg -> meso)  = {lift(A, B):.4f}")
print(f"LEVERAGE            = {leverage(A, B):.4f}")
print(f"CONVICTION  A->B        = {conviction(A, B):.4f}")

SUP(veg)           = 0.2000
SUP(meso)           = 0.2000
SUP(veg ∪ meso)    = 0.0000

CONF(veg -> meso)  = 0.0000
LIFT(veg -> meso)  = 0.0000
LEVERAGE            = -0.0400
CONVICTION  A->B        = 0.8000


*Negativna asocijacija, tj. što ne ide zajedno*

In [4]:
transactions

[{'kruh', 'maslac', 'mlijeko'},
 {'kava', 'kruh'},
 {'krafna', 'kruh'},
 {'kruh', 'čokolada'},
 {'kruh', 'mlijeko'},
 {'kava', 'krafna', 'kruh'},
 {'jogurt', 'kruh'},
 {'kruh', 'sir'},
 {'kruh', 'čaj'},
 {'kava', 'kruh'},
 {'kavijar', 'šampanjac'},
 {'veganski_proizvod'},
 {'veganski_proizvod'},
 {'meso'},
 {'meso'}]

### 1.2. Vizualizacija FP stabla

In [5]:
from collections import Counter

# Support pojedinačnih artikala
item_counts = Counter()

for t in transactions:
    item_counts.update(t)

print(item_counts)

# Sortiranje itema unutar transakcija prema potpori
ordered_transactions = []

for t in transactions:
    ordered = sorted(
        t,
        key=lambda item: (-item_counts[item], item)
    )
    ordered_transactions.append(ordered)

print("\nSortirane transakcije:")
for t in ordered_transactions:
    print(t)


# Jednostavni FP-tree - pretty print
class FPNode:
    def __init__(self, item):
        self.item = item
        self.count = 1
        self.children = {}

    def add_child(self, item):
        if item in self.children:
            self.children[item].count += 1
        else:
            self.children[item] = FPNode(item)

        return self.children[item]


root = FPNode("ROOT")
root.count = 0

for transaction in ordered_transactions:
    current = root

    for item in transaction:
        current = current.add_child(item)

print("\nFP-tree:")

def print_tree(node, prefix="", is_last=True):

    if node.item != "ROOT":

        connector = "└── " if is_last else "├── "

        print(
            prefix
            + connector
            + f"{node.item} ({node.count})"
        )

        prefix += "    " if is_last else "│   "

    children = list(node.children.values())

    for i, child in enumerate(children):

        is_last_child = (i == len(children) - 1)

        print_tree(child, prefix, is_last_child)


print("FP-tree:\n")

print("ROOT")

children = list(root.children.values())

for i, child in enumerate(children):

    is_last_child = (i == len(children) - 1)

    print_tree(child, "", is_last_child)


Counter({'kruh': 10, 'kava': 3, 'mlijeko': 2, 'krafna': 2, 'veganski_proizvod': 2, 'meso': 2, 'maslac': 1, 'čokolada': 1, 'jogurt': 1, 'sir': 1, 'čaj': 1, 'kavijar': 1, 'šampanjac': 1})

Sortirane transakcije:
['kruh', 'mlijeko', 'maslac']
['kruh', 'kava']
['kruh', 'krafna']
['kruh', 'čokolada']
['kruh', 'mlijeko']
['kruh', 'kava', 'krafna']
['kruh', 'jogurt']
['kruh', 'sir']
['kruh', 'čaj']
['kruh', 'kava']
['kavijar', 'šampanjac']
['veganski_proizvod']
['veganski_proizvod']
['meso']
['meso']

FP-tree:
FP-tree:

ROOT
├── kruh (10)
│   ├── mlijeko (2)
│   │   └── maslac (1)
│   ├── kava (3)
│   │   └── krafna (1)
│   ├── krafna (1)
│   ├── čokolada (1)
│   ├── jogurt (1)
│   ├── sir (1)
│   └── čaj (1)
├── kavijar (1)
│   └── šampanjac (1)
├── veganski_proizvod (2)
└── meso (2)


## 2. Linux - analiza logova
https://github.com/logpai/loghub/blob/master/Linux/Linux_2k.log

### 2.1. Ručna izrada pravila i metrike

In [6]:
import re
from collections import defaultdict


with open("Linux_2k.log", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

# ------------------------
# Pretvaranje logova u transakcije
# Jedna minuta = jedna transakcija
# ------------------------

transactions_dict = defaultdict(set)

for line in lines:

    timestamp = " ".join(line.split()[:3])[:12]

    items = set()

    # ručno mapiranje događaja
    if "authentication failure" in line:
        items.add("auth_failure")

    if "user unknown" in line:
        items.add("user_unknown")

    if "session opened" in line:
        items.add("session_open")

    if "session closed" in line:
        items.add("session_close")

    if "root" in line:
        items.add("root_activity")

    if "sshd" in line:
        items.add("sshd")

    if "su(" in line:
        items.add("su_command")

    if "ALERT" in line:
        items.add("alert")

    transactions_dict[timestamp].update(items)

transactions = list(transactions_dict.values())

print("Broj transakcija:", len(transactions))

print("\nPrimjeri transakcija:\n")

for t in transactions[:10]:
    print(t)

# Mjere
N = len(transactions)

def support(itemset):
    return sum(itemset.issubset(t) for t in transactions) / N

def confidence(A, B):
    return support(A | B) / support(A)

def lift(A, B):
    return support(A | B) / (support(A) * support(B))

# ------------------------
# Primjeri pravila
# ------------------------

rules = [
    ({"user_unknown"}, {"auth_failure"}),
    ({"auth_failure"}, {"sshd"}),
    ({"session_open"}, {"session_close"}),
    ({"root_activity"}, {"auth_failure"}),
]

print("\nAsocijativna pravila:\n")

for A, B in rules:

    print(f"{A} -> {B}")

    print(f"support     = {support(A | B):.3f}")
    print(f"confidence  = {confidence(A, B):.3f}")
    print(f"lift        = {lift(A, B):.3f}")

    print()

Broj transakcija: 235

Primjeri transakcija:

{'auth_failure', 'sshd', 'user_unknown'}
{'auth_failure', 'sshd', 'root_activity'}
{'alert', 'session_open', 'session_close', 'su_command'}
{'session_open', 'session_close', 'su_command'}
{'auth_failure', 'sshd', 'user_unknown'}
{'auth_failure', 'sshd', 'user_unknown'}
{'auth_failure', 'sshd', 'user_unknown'}
{'auth_failure', 'sshd', 'user_unknown'}
{'alert', 'session_open', 'session_close', 'su_command'}
{'session_open', 'session_close', 'su_command'}

Asocijativna pravila:

{'user_unknown'} -> {'auth_failure'}
support     = 0.077
confidence  = 1.000
lift        = 3.790

{'auth_failure'} -> {'sshd'}
support     = 0.260
confidence  = 0.984
lift        = 3.351

{'session_open'} -> {'session_close'}
support     = 0.387
confidence  = 0.968
lift        = 2.420

{'root_activity'} -> {'auth_failure'}
support     = 0.170
confidence  = 0.930
lift        = 3.526



*Svaki put kad se pojavi user_unknown, događa se i auth_failure, događaj je rijedak, međutim authentication failure pojavljuje se gotovo 4 puta češće uz unknown user nego što bismo očekivali slučajno (mjera lift).*

*root_activity -> auth_failure; root aktivnosti su snažno povezane s failed authentication događajima što
može ukazivati na: brute-force pokušaje, privilege escalation, administrativne aktivnosti, ili sigurnosne incidente*

### 2.2 FP-Growth

- koristi knjižnicu mlxtend (```pip install mlxtend```)

In [9]:
import pandas as pd
from collections import defaultdict

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules



with open("Linux_2k.log", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()


transactions_dict = defaultdict(set)

for line in lines:

    timestamp = " ".join(line.split()[:3])[:12]

    items = set()

    if "authentication failure" in line:
        items.add("auth_failure")

    if "user unknown" in line:
        items.add("user_unknown")

    if "session opened" in line:
        items.add("session_open")

    if "session closed" in line:
        items.add("session_close")

    if "root" in line:
        items.add("root_activity")

    if "sshd" in line:
        items.add("sshd")

    if "su(" in line:
        items.add("su_command")

    if "ALERT" in line:
        items.add("alert")

    if items:
        transactions_dict[timestamp].update(items)

transactions = list(transactions_dict.values())

print("Broj transakcija:", len(transactions))

# -----------------------------------
# Edukativni FP-tree prikaz
# -----------------------------------

class FPNode:

    def __init__(self, item):
        self.item = item
        self.count = 1
        self.children = {}

    def add_child(self, item):

        if item in self.children:
            self.children[item].count += 1
        else:
            self.children[item] = FPNode(item)

        return self.children[item]


def build_simple_fp_tree(transactions):

    item_counts = Counter()

    for transaction in transactions:
        item_counts.update(transaction)

    ordered_transactions = []

    for transaction in transactions:
        ordered = sorted(
            transaction,
            key=lambda item: (-item_counts[item], item)
        )
        ordered_transactions.append(ordered)

    root = FPNode("ROOT")
    root.count = 0

    for transaction in ordered_transactions:

        current = root

        for item in transaction:
            current = current.add_child(item)

    return root, item_counts, ordered_transactions


def print_tree(node, prefix="", is_last=True):

    if node.item != "ROOT":

        connector = "└── " if is_last else "├── "

        print(
            prefix
            + connector
            + f"{node.item} ({node.count})"
        )

        prefix += "    " if is_last else "│   "

    children = list(node.children.values())

    for i, child in enumerate(children):

        is_last_child = i == len(children) - 1
        print_tree(child, prefix, is_last_child)


fp_root, item_counts, ordered_transactions = build_simple_fp_tree(transactions)

print("\nSupport count pojedinačnih itema:\n")
for item, count in item_counts.most_common():
    print(f"{item}: {count}")

print("\nPrvih 10 sortiranih transakcija:\n")
for transaction in ordered_transactions[:10]:
    print(transaction)

print("\nEdukativni FP-tree prikaz:\n")
print("ROOT")

children = list(fp_root.children.values())

for i, child in enumerate(children):
    is_last_child = i == len(children) - 1
    print_tree(child, "", is_last_child)




# One-hot encoding
te = TransactionEncoder()

encoded = te.fit(transactions).transform(transactions)

df = pd.DataFrame(encoded, columns=te.columns_)

print("\nTablični prikaz:\n")
print(df.head())


# -----------------------------------
# FP-growth
# -----------------------------------

frequent_itemsets = fpgrowth(
    df,
    min_support=0.05,
    use_colnames=True
)

print("\nČesti itemsetovi:\n")
print(frequent_itemsets.sort_values(
    by="support",
    ascending=False
))


rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules = rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

print("\nAsocijativna pravila:\n")

print(rules.sort_values(
    by="lift",
    ascending=False
))

Broj transakcija: 164

Support count pojedinačnih itema:

session_open: 94
session_close: 94
su_command: 87
sshd: 69
auth_failure: 62
root_activity: 43
alert: 43
user_unknown: 18

Prvih 10 sortiranih transakcija:

['sshd', 'auth_failure', 'user_unknown']
['sshd', 'auth_failure', 'root_activity']
['session_close', 'session_open', 'su_command', 'alert']
['session_close', 'session_open', 'su_command']
['sshd', 'auth_failure', 'user_unknown']
['sshd', 'auth_failure', 'user_unknown']
['sshd', 'auth_failure', 'user_unknown']
['sshd', 'auth_failure', 'user_unknown']
['session_close', 'session_open', 'su_command', 'alert']
['session_close', 'session_open', 'su_command']

Edukativni FP-tree prikaz:

ROOT
├── sshd (61)
│   └── auth_failure (61)
│       ├── user_unknown (17)
│       └── root_activity (40)
├── session_close (94)
│   ├── session_open (91)
│   │   ├── su_command (85)
│   │   │   └── alert (39)
│   │   └── sshd (6)
│   ├── sshd (1)
│   ├── root_activity (1)
│   └── su_command (1)
├──

### 2.3 HUIM i EFIM
- koristi knjižnicu PAMI (```pip install pami```)

In [8]:
from collections import defaultdict, Counter
from PAMI.highUtilityPattern.basic import EFIM

with open("Linux_2k.log", "r", encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()

# Dodjela težina (npr. sigurnosna važnost događaja)
# ovo moramo ručno jer u linux log datasetu toga nema


event_weights = {
    "auth_failure": 5,
    "user_unknown": 7,
    "session_open": 1,
    "session_close": 1,
    "root_activity": 10,
    "sshd": 3,
    "su_command": 8,
    "alert": 15
}

item_to_id = {
    item: i + 1 for i, item in enumerate(event_weights)
}

id_to_item = {
    v: k for k, v in item_to_id.items()
}

transactions_dict = defaultdict(list)

for line in lines:
    # jedna minuta = jedna transakcija
    timestamp = " ".join(line.split()[:3])[:12]

    if "authentication failure" in line:
        transactions_dict[timestamp].append("auth_failure")

    if "user unknown" in line:
        transactions_dict[timestamp].append("user_unknown")

    if "session opened" in line:
        transactions_dict[timestamp].append("session_open")

    if "session closed" in line:
        transactions_dict[timestamp].append("session_close")

    if "root" in line:
        transactions_dict[timestamp].append("root_activity")

    if "sshd" in line:
        transactions_dict[timestamp].append("sshd")

    if "su(" in line:
        transactions_dict[timestamp].append("su_command")

    if "ALERT" in line:
        transactions_dict[timestamp].append("alert")


with open("linux_huim.txt", "w", encoding="utf-8") as out:

    for events in transactions_dict.values():

        if not events:
            continue

        counts = Counter(events)

        items = []
        utilities = []

        for event, count in counts.items():
            item_id = item_to_id[event]
            utility = event_weights[event] * count

            items.append(item_id)
            utilities.append(utility)

        # sort zbog urednosti
        paired = sorted(zip(items, utilities))
        items = [x[0] for x in paired]
        utilities = [x[1] for x in paired]

        transaction_utility = sum(utilities)

        out.write(
            " ".join(map(str, items))
            + ":"
            + str(transaction_utility)
            + ":"
            + " ".join(map(str, utilities))
            + "\n"
        )

print("HUIM datoteka spremljena.")
print("Mapiranje itema:")
print(item_to_id)


model = EFIM.EFIM(
    iFile="linux_huim.txt",
    minUtil=30,
    sep=" "
)

model.mine()

patterns = model.getPatterns()

for pattern, utility in patterns.items():

    item_ids = pattern.split()
    item_names = [id_to_item[int(i)] for i in item_ids]

    print(item_names, "-> utility =", utility)

HUIM datoteka spremljena.
Mapiranje itema:
{'auth_failure': 1, 'user_unknown': 2, 'session_open': 3, 'session_close': 4, 'root_activity': 5, 'sshd': 6, 'su_command': 7, 'alert': 8}
High Utility patterns were generated successfully using EFIM algorithm
['alert'] -> utility = 645
['alert', 'su_command'] -> utility = 1209
['alert', 'su_command', 'session_close'] -> utility = 1248
['alert', 'su_command', 'session_close', 'session_open'] -> utility = 1287
['alert', 'su_command', 'session_open'] -> utility = 1248
['alert', 'session_close'] -> utility = 624
['alert', 'session_close', 'session_open'] -> utility = 663
['alert', 'session_open'] -> utility = 624
['user_unknown'] -> utility = 819
['user_unknown', 'auth_failure'] -> utility = 1409
['user_unknown', 'auth_failure', 'sshd'] -> utility = 2096
['user_unknown', 'sshd'] -> utility = 1511
['su_command'] -> utility = 1376
['su_command', 'session_close'] -> utility = 1454
['su_command', 'session_close', 'session_open'] -> utility = 1530
['su